In [15]:
import os
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

# setup stuff.  This cell just supports the workbook, you can ignore it
TEST_MOUNTED_FOLDER = os.path.join(os.path.dirname(os.getcwd()), "tests", "test_mounted_folder")

def show_file(path):
    """Show contents of a file with line prefix."""
    with open(path) as file:
        contents = file.read()
    with_bar = "\n  | ".join(contents.split("\n"))
    print(f"\n### SHOWING {path!r}\n  | {with_bar}\n\n")

### Imports

In [16]:
# This is the key class and key function for DVC-DAT
from dvc_dat import Dat, do

# Creating Dats
A dat is simply a configuration dict stored in its own folder.


In [17]:
# Here is a dat created with an implicitly defined path.
spec = {"foo": "bar", "dat": {"my_key1": "my_val1", "my_key2": "my_val2"}}
anon_dat1=Dat.create(spec=spec)
anon_dat2=Dat.create(spec=spec)

print(f"dat folder = {anon_dat2.get_path()!r}")
os.system(f"ls -1 {anon_dat2.get_path()}")

anon_dat2

dat folder = '/Users/oblinger/ob/proj/dvc-dat/tests/test_sync_folder/anonymous/Dat_40'
_spec_.yaml


<Dat: tests/test_sync_folder/anonymous/Dat_40>

#### Creating a dat using a templated path

In [18]:

dat0 = Dat.create(path="{YYYY}-{MM}-{DD}/my_dat", spec=spec, overwrite=True)

print(f"dat folder = {dat0.get_path()!r}")
dat0

dat folder = '/Users/oblinger/ob/proj/dvc-dat/tests/test_sync_folder/2025-11-28/my_dat'


<Dat: tests/test_sync_folder/2025-11-28/my_dat>

## Accessing its expected attributes.

In [19]:
print(f"dat.my_key1 = {Dat.get(dat0, 'dat.my_key1')!r}")
dat0.get_spec()

dat.my_key1 = 'my_val1'


{'dat': {'kind': 'Dat',
  'base': None,
  'do': None,
  'args': None,
  'kwargs': None,
  'my_key1': 'my_val1',
  'my_key2': 'my_val2'},
 'foo': 'bar'}

### The three path accessors
- `dat.get_path()`  # this is the full path to the dat's folder
- `dat.get_path_name()`  # this is the 'name' of the dat -- it is the suffix of the path relative to the dat_folder
- `dat.get_path_tail()`  # this is the last segment of the path -- this tail is the shortname or informal name for this dat

In [20]:
print(f"path = {dat0.get_path()!r}")
print(f"name = {dat0.get_path_name()!r}")
print(f"shortname = {dat0.get_path_tail()!r}")

path = '/Users/oblinger/ob/proj/dvc-dat/tests/test_sync_folder/2025-11-28/my_dat'
name = 'tests/test_sync_folder/2025-11-28/my_dat'
shortname = 'my_dat'


In [21]:
# Clean up any leftover data from previous runs
import shutil
cleanup_path = os.path.join(Dat.manager.main_sync_folder, "new_location")
if os.path.exists(cleanup_path):
    shutil.rmtree(cleanup_path)

dat2 = dat0.move("new_location/the_dat")
dat3 = dat2.copy("new_location/the_dat3")
print(f"dat2 path = {dat2.get_path()!r}")
print(f"dat3 path_name = {dat3.get_path_name()!r}")
print(f"dat3 exists: {Dat.manager.exists('new_location/the_dat3')!r}")
dat2.delete()
dat3.delete()

dat2 path = '/Users/oblinger/ob/proj/dvc-dat/tests/test_sync_folder/new_location/the_dat'
dat3 path_name = 'tests/test_sync_folder/new_location/the_dat3'
dat3 exists: True


True

# Loading and Saving to DVC

    my_dat = dat.get_path_name()  # string that can be used later to get this Dat back
    dat.flag()                     # flags this Dat to have a version saved to the DVC backing store.
    % ./do flag DAT_NAME           # manually flags a dat to be saved by DVC
    % edit flagged_dats.txt        # edit this file to remove dats you don't want to save
    % ./do push                     # Processes all flagged dats (adds to DVC, to git, commits & pushes)
    dat = Dat.load(my_dat)       # loads same data onto any other computer

The "do push" command will:
 (1) add the indicated folder to DVC and push the folder's content to the backing store
 (2) add the resulting .dvc file to git
 (3) commit the change to git
 (4) push the change to the git repository

Anytime before the do push, the flagged_dat.txt file can be edited to remove Dats that the user does not want saved to the backing store.